In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import os
 
# Path to your Who&When JSON file — edit this.
DATASET_DIR = "./ww"
SUBSET = "hand-crafted"
 
# Model names used in the evaluation plan.
MODEL_LLAMA = "/data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct"
MODEL_QWEN  = "/data/hoang/resources/models/Qwen/Qwen3-8B" 

# Which model to load in this session.
MODEL_NAME = MODEL_LLAMA
 
DEVICE = "1"   # "cpu" for a CPU-only sanity pass (slow)

In [2]:
from core.data import load_dataset, build_context, select_context
from utils.graph import get_dependency_dict, derive_llm_inputs
import math

from transformers import PreTrainedModel, PreTrainedTokenizer
from torch import Tensor
from core.gradnorm import _ntp_loss


trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj = trajectories[11]
len(traj.history)
deps = get_dependency_dict(derive_llm_inputs(traj.history))
# traj.history

In [3]:
deps

{0: [],
 1: [0],
 2: [0, 1],
 3: [2],
 4: [0, 1, 3],
 5: [0, 1, 3, 4],
 6: [5],
 7: [5],
 8: [0, 1, 3, 4, 6],
 9: [0, 1, 3, 4, 6, 8],
 10: [9],
 11: [9],
 12: [0, 1, 3, 4, 6, 8, 10],
 13: [0, 1, 3, 4, 6, 8, 10, 12],
 14: [13],
 15: [13],
 16: [0, 1, 3, 4, 6, 8, 10, 12, 14],
 17: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16],
 18: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16],
 19: [0, 1, 3, 4, 6, 8, 10, 12, 14, 16]}

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"\nLoading tokeniser: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# print(f"Loading model ({MODEL_NAME}) → {DEVICE}")
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype  = torch.bfloat16,
#     device_map   = {"": int(DEVICE)},
# )
# model.eval()
# n_params = sum(p.numel() for p in model.parameters())

# print(f"  {n_params / 1e9:.2f}B parameters loaded.\n")


Loading tokeniser: /data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct


In [7]:
tokenizer

TokenizersBackend(name_or_path='/data/hoang/resources/models/meta-llama/Llama-3.1-8B-Instruct', vocab_size=128000, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False,

In [6]:
messages = [
  {"role": "system", "content": "You are a bot that responds to weather queries."},
  {"role": "user", "content": "Hey, what's the temperature in Paris right now?"}
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11164, 430, 31680, 311, 9282, 20126, 13, 128009, 128006, 882, 128007, 271, 19182, 11, 1148, 596, 279, 9499, 304, 12366, 1314, 1457, 30, 128009, 128006, 78191, 128007, 271], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
def pad_encoded(encoded: dict[Tensor, int], max_tokens=4096):
    padded = tokenizer.pad(
        [{"input_ids": ids} for ids in encoded["input_ids"]],
        return_tensors="pt",
        padding_side="left",
        padding="max_length",
        max_length=max_tokens
    )
    padded_ids, attention_mask = padded["input_ids"], padded["attention_mask"]
    num_padded = int(attention_mask.shape[1] - attention_mask.sum(dim=1))
    padded_ctx_len = encoded["ctx_len"] + num_padded
    return {
        "input_ids": padded_ids, 
        "attention_mask": attention_mask, 
        "ctx_len": padded_ctx_len
    }

In [15]:
# ── Clean up memory ─────────────────────────────────────────────
def memory_accounting():
    device = f"cuda:{DEVICE}"
    before_mb = torch.cuda.memory_reserved(device) / 1e6
    torch.cuda.empty_cache()
    after_mb = torch.cuda.memory_reserved(device) / 1e6
    print(f"[{device}] reserved: {before_mb:.1f} MB → {after_mb:.1f} MB "
          f"(freed {before_mb - after_mb:.1f} MB)")
    print(f"[{device}] allocated: {torch.cuda.memory_allocated(device) / 1e6:.1f} MB")


## Standard Gradnorm

In [8]:
def gradnorm_standard(
    model:          PreTrainedModel,
    input_ids:      Tensor,
    attention_mask: Tensor,
    ctx_len:        int,
    normalize:      bool = True,
) -> dict:
    # ── Compute gradients ────────────────────────────────────────────
    logits = model(
        input_ids, 
        attention_mask, 
        use_cache=False
    ).logits
    loss   = _ntp_loss(logits, input_ids, ctx_len)
    loss.backward()

    # ── Compute score ────────────────────────────────────────────────
    target_weights = list(model.model.layers) + [model.lm_head]
    n_layers = len(model.model.layers)
    module_names = [f"layer_{i}" for i in range(n_layers)] + ["lm_head"]

    statistics = {}
    for name, module in zip(module_names, target_weights):
        n_params = sum(p.numel() for p in module.parameters())
        l1_norm  = sum(p.grad.detach().abs().sum().item() for p in module.parameters())
        l2_norm  = math.sqrt(
            sum(p.grad.detach().norm(2).item() ** 2 for p in module.parameters())
        )
        if normalize:
            l1_norm /= n_params
            l2_norm /= n_params

        statistics[name] = {
            "l1_norm": l1_norm,
            "l2_norm": l2_norm,
        }

    # ── Cleanup ──────────────────────────────────────────────────────
    for p in model.parameters():
        p.grad = None
    memory_accounting()
    
    return statistics

In [ ]:

trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj = trajectories[11]
context_builder = build_context
MAX_TOKENS = 4096

tests = []

for step_idx in range(len(traj.history)):
    # ── Tokenise ────────────────────────────────────────────────────
    encoded   = context_builder(
        traj.history, 
        step_idx, 
        tokenizer,
        max_tokens=MAX_TOKENS
    )
    # encoded = pad_encoded(encoded)

    input_ids = encoded["input_ids"].to(f"cuda:{DEVICE}")   # (1, seq_len)
    attention_mask = encoded.get("attention_mask", None)
    ctx_len   = encoded["ctx_len"]
    seq_len   = input_ids.shape[1]

    print(f"seq_len: {seq_len}, ctx_len: {ctx_len}")

    statistics = gradnorm_standard(
        model, input_ids, attention_mask, ctx_len
    )
    if isinstance(attention_mask, Tensor):
        attention_mask = attention_mask.tolist()
    elif attention_mask is None:
        attention_mask = None
    else:
        raise NotImplementedError()
    
    out = {
        "metadata": {
            "filename": traj.filename,
            "step_idx": step_idx,
            "input_ids": input_ids.tolist(),
            "attention_mask": attention_mask,
            "ctx_len": ctx_len,
        },
        "statistics": statistics
    }
    tests.append(out)

seq_len: 76, ctx_len: 8
[cuda:1] CUDA allocated : 8085.1 MB
[cuda:1] CUDA reserved  : 8134.9 MB
seq_len: 800, ctx_len: 70
[cuda:1] CUDA allocated : 8305.3 MB
[cuda:1] CUDA reserved  : 8350.9 MB
seq_len: 1091, ctx_len: 794
[cuda:1] CUDA allocated : 8393.5 MB
[cuda:1] CUDA reserved  : 8443.1 MB
seq_len: 385, ctx_len: 299
[cuda:1] CUDA allocated : 8179.4 MB
[cuda:1] CUDA reserved  : 8227.1 MB
seq_len: 2154, ctx_len: 874
[cuda:1] CUDA allocated : 8716.6 MB
[cuda:1] CUDA reserved  : 8766.1 MB
seq_len: 2469, ctx_len: 2148
[cuda:1] CUDA allocated : 8812.8 MB
[cuda:1] CUDA reserved  : 8860.5 MB
seq_len: 408, ctx_len: 323
[cuda:1] CUDA allocated : 8186.0 MB
[cuda:1] CUDA reserved  : 8235.5 MB
seq_len: 347, ctx_len: 323
[cuda:1] CUDA allocated : 8167.4 MB
[cuda:1] CUDA reserved  : 8216.6 MB
seq_len: 3367, ctx_len: 2227
[cuda:1] CUDA allocated : 9085.4 MB
[cuda:1] CUDA reserved  : 9133.1 MB
seq_len: 3684, ctx_len: 3361
[cuda:1] CUDA allocated : 9181.9 MB
[cuda:1] CUDA reserved  : 9229.6 MB
seq_le

In [16]:
memory_accounting()

[cuda:1] reserved: 26831.0 MB → 16502.5 MB (freed 10328.5 MB)
[cuda:1] allocated: 16398.6 MB


In [ ]:
import json
os.makedirs("tests", exist_ok=True)
with open("tests/standard-unpadded.json", "w") as f:
    json.dump(tests, f, indent=4)

In [ ]:
tests1 = json.load(open("tests/standard-padded.json"))
tests2 = json.load(open("tests/standard-unpadded.json"))

In [ ]:
layer_names = [key for key in tests1[0]["statistics"]]

name = layer_names[-4]
stat1 = tests1[0]["statistics"][name]
stat2 = tests2[0]["statistics"][name]
name, stat1, stat2

('layer_33',
 {'l1_norm': 0.0004390267500165658, 'l2_norm': 1.485981021025087e-07},
 {'l1_norm': 0.00043918508093454827, 'l2_norm': 1.4851277199824687e-07})

In [ ]:
print("Well padded and unpadded are roughly the same.")
print("They are ready to be testcases for optimization")

Well padded and unpadded are roughly the same.
They are ready to be testcases for optimization


## Memory optimization

In [7]:
memory_accounting()

[cuda:1] allocated: 16381.5 MB → 16381.5 MB (freed 0.0 MB)
[cuda:1] reserved: 16385.0 MB


In [8]:
import math
import math
import torch
from torch import Tensor
from transformers import PreTrainedModel

def gradnorm_hooked(
    model:          PreTrainedModel,
    input_ids:      Tensor,
    attention_mask: Tensor,
    ctx_len:        int,
    normalize:      bool = True,
) -> dict:
    # ── 1. Fix ACTIVATION memory ─────────────────────────────────────
    # HF silently ignores gradient checkpointing if model is in eval mode!
    was_training = model.training
    model.train()
    
    # Ensure inputs require grad so checkpointing triggers correctly
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
        
    model.gradient_checkpointing_enable()

    # ── Build module → name mapping ──────────────────────────────────
    target_modules = {module: name for name, module in zip(
        [f"layer_{i}" for i in range(len(model.model.layers))] + ["lm_head"],
        list(model.model.layers) + [model.lm_head],
    )}

    # ── 2. Fix GRADIENT memory via Post-Accumulate Hooks ─────────────
    statistics = {}
    handles = []
    hooked_params = set() # Prevents double-counting tied weights (e.g. Qwen embeddings)
    
    # Pre-allocate stats on GPU to prevent CPU-GPU sync bottlenecks!
    device = next(model.parameters()).device

    for module, name in target_modules.items():
        statistics[name] = {
            "l1_norm": torch.tensor(0.0, device=device, dtype=torch.float64), 
            "l2_norm_sq": torch.tensor(0.0, device=device, dtype=torch.float64), 
            "n_params": 0
        }
        
        for p in module.parameters():
            if not p.requires_grad or p in hooked_params:
                continue
            hooked_params.add(p)
            
            entry = statistics[name]
            entry["n_params"] += p.numel()

            def make_stat_hook(entry_dict):
                # PyTorch 2.1+ post-accumulate hook receives the parameter itself
                def hook(param):
                    if param.grad is not None:
                        with torch.no_grad():
                            # Cast to float32/64 to prevent bf16 overflow when squaring
                            grad_f32 = param.grad.float()
                            entry_dict["l1_norm"]    += grad_f32.abs().sum().double()
                            entry_dict["l2_norm_sq"] += grad_f32.square().sum().double()
                        
                        # ✨ IMMEDIATELY free the gradient memory! ✨
                        param.grad = None
                return hook

            h = p.register_post_accumulate_grad_hook(make_stat_hook(entry))
            handles.append(h)

    # ── 3. Catch untracked parameters (embeddings, layernorm) ────────
    # So they don't silently leak VRAM
    def clear_hook(param):
        param.grad = None

    for p in model.parameters():
        if p.requires_grad and p not in hooked_params:
            h = p.register_post_accumulate_grad_hook(clear_hook)
            handles.append(h)
            hooked_params.add(p)

    # ── Forward + backward ───────────────────────────────────────────
    model.zero_grad(set_to_none=True)
    
    logits = model(
        input_ids, attention_mask, use_cache=False,
    ).logits
    
    loss = _ntp_loss(logits, input_ids, ctx_len)
    
    # As backward runs, gradients are instantiated, recorded, and instantly destroyed!
    loss.backward()

    # ── Cleanup ──────────────────────────────────────────────────────
    for h in handles:
        h.remove()
        
    model.gradient_checkpointing_disable()
    if not was_training:
        model.eval()

    # ── 4. Pull metrics to CPU exactly ONCE ──────────────────────────
    for name, stats in statistics.items():
        n_params = stats.pop("n_params")
        
        l1_val = stats.pop("l1_norm").item()
        l2_val = math.sqrt(stats.pop("l2_norm_sq").item())
        
        if normalize and n_params > 0:
            l1_val /= n_params
            l2_val /= n_params
            
        stats["l1_norm"] = l1_val
        stats["l2_norm"] = l2_val

    # Double check no stray gradients remain
    model.zero_grad(set_to_none=True)

    return statistics

In [17]:
trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj_idx = 11
step_idx = 15
traj = trajectories[traj_idx]
context_builder = build_context
MAX_TOKENS = 8192

# ── Tokenise ────────────────────────────────────────────────────
encoded   = context_builder(
    traj.history, 
    step_idx, 
    tokenizer,
    max_tokens=MAX_TOKENS
)
encoded = pad_encoded(encoded, max_tokens=MAX_TOKENS)

input_ids = encoded["input_ids"].to(f"cuda:{DEVICE}")   # (1, seq_len)
attention_mask = encoded.get("attention_mask", None)
ctx_len   = encoded["ctx_len"]
seq_len   = input_ids.shape[1]
print(f"seq_len: {seq_len}, ctx_len: {ctx_len}")

statistics = gradnorm_hooked(
    model, input_ids, attention_mask, ctx_len
)

seq_len: 8192, ctx_len: 8085


In [14]:
torch.cuda.memory_reserved("cuda:1") / 1e6

26830.962688

In [10]:
memory_accounting()

[cuda:1] allocated: 16398.6 MB → 16398.6 MB (freed 0.0 MB)
[cuda:1] reserved: 16502.5 MB


In [ ]:
for p in model.parameters():
    # p.grad = None
    if p.grad is not None: print(p.grad)

In [ ]:
statistics = gradnorm_standard(
    model, input_ids, attention_mask, ctx_len
)

In [26]:
memory_accounting()
for p in model.parameters():
    p.grad = None

[cuda:1] allocated: 8062.0 MB → 8062.0 MB (freed 0.0 MB)
[cuda:1] reserved: 8099.2 MB


In [19]:

trajectories  = load_dataset(DATASET_DIR, subset=SUBSET)
traj = trajectories[11]
context_builder = build_context
MAX_TOKENS = 4096

tests = []

for step_idx in range(len(traj.history)):
    # ── Tokenise ────────────────────────────────────────────────────
    encoded   = context_builder(
        traj.history, 
        step_idx, 
        tokenizer,
        max_tokens=MAX_TOKENS
    )
    encoded = pad_encoded(encoded)

    input_ids = encoded["input_ids"].to(f"cuda:{DEVICE}")   # (1, seq_len)
    attention_mask = encoded.get("attention_mask", None)
    ctx_len   = encoded["ctx_len"]
    seq_len   = input_ids.shape[1]

    print(f"seq_len: {seq_len}, ctx_len: {ctx_len}")

    statistics = gradnorm_hooked(
        model, input_ids, attention_mask, ctx_len
    )
    if isinstance(attention_mask, Tensor):
        attention_mask = attention_mask.tolist()
    elif attention_mask is None:
        attention_mask = None
    else:
        raise NotImplementedError()
    
    out = {
        "metadata": {
            "filename": traj.filename,
            "step_idx": step_idx,
            "input_ids": input_ids.tolist(),
            "attention_mask": attention_mask,
            "ctx_len": ctx_len,
        },
        "statistics": statistics
    }
    tests.append(out)

seq_len: 4096, ctx_len: 4028
seq_len: 4096, ctx_len: 3366
seq_len: 4096, ctx_len: 3799
seq_len: 4096, ctx_len: 4010
seq_len: 4096, ctx_len: 2816
seq_len: 4096, ctx_len: 3775
seq_len: 4096, ctx_len: 4011
seq_len: 4096, ctx_len: 4072
seq_len: 4096, ctx_len: 2956
seq_len: 4096, ctx_len: 3773
seq_len: 4096, ctx_len: 4031
seq_len: 4096, ctx_len: 4071
seq_len: 4096, ctx_len: 2544
seq_len: 4096, ctx_len: 3838
seq_len: 4096, ctx_len: 4026
seq_len: 4096, ctx_len: 3989
seq_len: 4096, ctx_len: 3765
seq_len: 4096, ctx_len: 3865
seq_len: 4096, ctx_len: 4073
seq_len: 4096, ctx_len: 4042


In [ ]:
import json
tests_padded = json.load(open("tests/standard-padded.json"))

In [25]:
layer_names = [key for key in tests_padded[0]["statistics"]]

name = layer_names[-15]
stat1 = tests[0]["statistics"][name]
stat2 = tests_padded[0]["statistics"][name]
name, stat1, stat2

('layer_22',
 {'l1_norm': 0.0007828110825263432, 'l2_norm': 2.586599180327348e-07},
 {'l1_norm': 0.0007833052754175692, 'l2_norm': 2.590100212218981e-07})

In [ ]:
for p in model.parameters():
    p.grad = None
memory_accounting()

[cuda:1] allocated: 40033.3 MB → 40033.3 MB (freed 0.0 MB)
[cuda:1] reserved: 45113.9 MB
